In [0]:
%python
import os 
print(os.listdir('/Volumes/bundestag_dev/bronze/raw_xml/21/'))

In [0]:
%python
from xml_fetcher import fetch_xml

In [0]:
%python
PROTOCOL_URLS = [
    "https://www.bundestag.de/resource/blob/1150798/21060.xml",
    "https://www.bundestag.de/resource/blob/1150624/21059.xml",
    "https://www.bundestag.de/resource/blob/1150344/21058.xml",
    "https://www.bundestag.de/resource/blob/1140642/21057.xml",
    "https://www.bundestag.de/resource/blob/1140512/21056.xml",
    "https://www.bundestag.de/resource/blob/1140144/21055.xml",
]

In [0]:
%python
from xml_fetcher import fetch_xml
from register_session_ingestion import register_xml_fetch

for url in PROTOCOL_URLS:
    try:
        file_path= fetch_xml(input_url =url,
                             output_directory = '/Volumes/bundestag_dev/bronze/raw_xml')
    except ValueError as e:
        print(f'Saving error: {e}')
    else:
        print("File succesfully saved")
        print(f"Filepath: {file_path}")

        try:
            register_xml_fetch(spark= spark, 
                               filepath = file_path, 
                               source_path = url)

        except Exception as e:
            print(f"Registration Error: {e}")
        else:
            print("File succesfully registered")
      

    



In [0]:
spark.sql("DELETE FROM bundestag_dev.bronze.raw_sessions;")

In [0]:
spark.sql("SELECT * FROM bundestag_dev.bronze.raw_sessions").show()

In [0]:
%pip install pandera
%pip install typing 
%pip install --upgrade typing_extensions
%restart_python

In [0]:
from session_extraction import session_extraction
from cleaning import cleaning_raw_speeches
from schemas_bronze import bronze_schema, silver_staging_spark_schema
import pandas as pd

# get xml file paths
## declaring raw sessions table as df
raw_sessions_df = spark.table("bundestag_dev.bronze.raw_sessions")
pending_sessions_df = raw_sessions_df.filter(raw_sessions_df.status != "processed")

# converting spark df to pandas df, selecting only file url column
# converting values to list
xml_data = pending_sessions_df.toPandas()[["filename","file_url"]].to_dict(orient="records")
  
# for each xml file extract speeches, clean, validate and save speeches
for xml in xml_data:
    filename = xml["filename"]
    file_url = xml["file_url"]
    # extract raw speeches
    raw_speeches_df = session_extraction(file_url)
    # clean raw speeches
    clean_speeches_df = cleaning_raw_speeches(raw_speeches_df)
    # validate cleaned speeches 
    validated_speeches_df = bronze_schema.validate(clean_speeches_df)

    # add source_path column
    validated_speeches_df["source_path"] = file_url
    
    # converting pandas df to spark df to leverage native spark 
    # delta table writing
    validated_spark_df = spark.createDataFrame(data=validated_speeches_df
                                               , schema=silver_staging_spark_schema)
    validated_spark_df.write.format("delta").mode("append").saveAsTable("bundestag_dev.silver_staging.speeches_staging")

    # update session as processed
    spark.sql(f"""
              UPDATE bundestag_dev.bronze.raw_sessions
              SET status = 'processed',
                  processed_at = current_timestamp()
              WHERE filename = '{filename}';
              """)
    



In [0]:
spark.sql("SELECT COUNT(*) FROM bundestag_dev.silver_staging.speeches_staging;").show()

In [0]:
spark.sql("DELETE FROM bundestag_dev.silver_staging.speeches_staging;")

In [0]:
spark.sql("SELECT * FROM bundestag_dev.silver_staging.speeches_staging LIMIT 10").toPandas().to_csv("/Volumes/bundestag_dev/bronze/raw_xml/speeches_sample.csv", index=False)

In [0]:
import pandas as pd

# get xml file paths
## declaring raw sessions table as df
raw_sessions_df = spark.table("bundestag_dev.bronze.raw_sessions")

# converting spark df to pandas df, selecting only file url column
# converting values to list
xml_urls = raw_sessions_df.toPandas()

In [0]:
for x in xml_urls:
    print(f"Filename:{x["filename"]}")
    print(f"Url:{x["file_url"]}")